# LeetCode 4105 — Repeated String Match II (Hard)

Bài toán LeetCode:

```python
class Solution:
    def repeatedStringMatch(self, a: str, b: str, x: int, y: int) -> bool:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4105 (Biweekly Contest 195 — câu 4, rating khoảng 2208).

Đề bài nghe rất đơn giản: *"`b` lặp `y` lần có nằm trong `a` lặp `x` lần không?"*. Cái khó là `x, y` lên tới `10^9` — không thể dựng chuỗi ra. Ta sẽ học cách **khai thác tính tuần hoàn (periodicity)** của chuỗi lặp:

- vị trí bắt đầu chỉ cần xét **theo modulo `|a|`**;
- hai chuỗi tuần hoàn khớp nhau trên một đoạn **đủ dài** thì khớp mãi mãi (chu kỳ chung `lcm`, rồi mạnh hơn: **định lý Fine–Wilf**);
- cuối cùng dùng **KMP** để kiểm tra trong thời gian tuyến tính.

---

## Sau notebook này, bạn sẽ nắm được

1. Chuỗi lặp, chỉ số theo modulo, `gcd`, `lcm`.
2. Brute Force (chỉ dùng được khi `x, y` nhỏ).
3. Vì sao chỉ cần thử `|a|` vị trí bắt đầu.
4. Vì sao chỉ cần so sánh `lcm(|a|, |b|)` ký tự, rồi chỉ `|a| + |b| - gcd(|a|, |b|)` ký tự.
5. Thuật toán KMP từ con số 0.
6. Các trường hợp biên (độ dài vượt quá), lỗi hay gặp, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho hai chuỗi chữ thường `a`, `b` và hai số nguyên `x`, `y`.

- `a * x` là `a` nối với chính nó đúng `x` lần.
- `b * y` là `b` nối với chính nó đúng `y` lần.

Trả về `True` nếu `b * y` là **substring** (chuỗi con liên tiếp) của `a * x`, ngược lại `False`.

---

## Ví dụ 1

```python
a = "ab", b = "abab", x = 2, y = 1   ->   True
```

`a * 2 = "abab"`, `b * 1 = "abab"` — trùng nhau.

## Ví dụ 2

```python
a = "abc", b = "bc", x = 2, y = 3   ->   False
```

`a * 2 = "abcabc"`, `b * 3 = "bcbcbc"` — không xuất hiện.

## Ví dụ 3

```python
a = "abcd", b = "cdab", x = 2, y = 1   ->   True
```

`"abcdabcd"` chứa `"cdab"` ở vị trí 2.

---

## Ràng buộc

```text
1 <= a.length, b.length <= 100
1 <= x, y <= 10^9
a, b chỉ gồm chữ cái tiếng Anh viết thường
```

`a * x` có thể dài `10^11` ký tự — không thể dựng ra bộ nhớ.

# 2. Kiến thức nền tảng

## 2.1. Chuỗi lặp và chỉ số theo modulo

Ký tự thứ `t` (đếm từ 0) của chuỗi `a` lặp vô hạn là `a[t % len(a)]`. Ta **không cần dựng chuỗi** để biết ký tự ở vị trí bất kỳ.

In [1]:
a = "abc"
n = len(a)
print((a * 4))
print("".join(a[t % n] for t in range(12)))     # giống hệt, không cần nhân chuỗi
print("ký tự thứ 10^11 của a lặp vô hạn:", a[10**11 % n])

abcabcabcabc
abcabcabcabc
ký tự thứ 10^11 của a lặp vô hạn: b


## 2.2. `gcd` và `lcm`

- `gcd(p, q)`: ước chung lớn nhất.
- `lcm(p, q) = p · q / gcd(p, q)`: bội chung nhỏ nhất.

Trong Python: `math.gcd`, `math.lcm` (Python 3.9+).

In [2]:
from math import gcd, lcm

for p, q in [(4, 6), (2, 4), (3, 5), (100, 99)]:
    print(f"gcd({p}, {q}) = {gcd(p, q):>3}, lcm({p}, {q}) = {lcm(p, q):>5}, p + q - gcd = {p + q - gcd(p, q)}")

gcd(4, 6) =   2, lcm(4, 6) =    12, p + q - gcd = 8
gcd(2, 4) =   2, lcm(2, 4) =     4, p + q - gcd = 4
gcd(3, 5) =   1, lcm(3, 5) =    15, p + q - gcd = 7
gcd(100, 99) =   1, lcm(100, 99) =  9900, p + q - gcd = 198


## 2.3. `itertools.cycle` và `islice`

`cycle(s)` lặp `s` vô hạn; `islice(it, k)` lấy `k` phần tử đầu. Kết hợp lại: lấy `k` ký tự đầu của `s` lặp vô hạn mà không phải đoán số lần lặp.

In [3]:
from itertools import cycle, islice

print("".join(islice(cycle("abc"), 7)))   # abcabca

abcabca


# 3. Cách 1 — Brute Force: dựng chuỗi thật

Dùng toán tử `in` của Python. Chỉ chạy được khi `x, y` nhỏ.

In [4]:
def match_bruteforce(a: str, b: str, x: int, y: int) -> bool:
    return (b * y) in (a * x)


print(match_bruteforce("ab", "abab", 2, 1))    # True
print(match_bruteforce("abc", "bc", 2, 3))     # False
print(match_bruteforce("abcd", "cdab", 2, 1))  # True

True
False
True


Với `x = 10^9`, `a * x` tốn tới 100 GB bộ nhớ. Ta cần suy luận thay vì dựng chuỗi.

---

# 4. Quan sát 1: chỉ cần thử `n` vị trí bắt đầu

Gọi `n = |a|`, `m = |b|`, `L = m · y` (độ dài `b * y`), `T = n · x` (độ dài `a * x`).

**Điều kiện cần:** `L <= T`, nếu không thì `False` ngay.

`b * y` xuất hiện tại vị trí `p` của `a * x` khi:

1. `p + L <= T` (không vượt quá cuối),
2. với mọi `t` từ `0` tới `L - 1`: `a[(p + t) % n] == (b * y)[t]`.

Điều kiện 2 **chỉ phụ thuộc `p % n`**. Còn điều kiện 1 càng dễ thỏa khi `p` càng nhỏ. Vậy nếu có vị trí `p` nào khớp, thì vị trí `p % n` (nhỏ hơn hoặc bằng) cũng khớp. Do đó:

> Chỉ cần thử `p = 0, 1, ..., min(n - 1, T - L)`.

Nhưng mỗi lần thử vẫn phải so sánh `L` ký tự — `L` có thể là `10^11`. Cần thêm một quan sát.

In [5]:
def match_offsets_full(a: str, b: str, x: int, y: int) -> bool:
    n, m = len(a), len(b)
    L, T = m * y, n * x
    if L > T:
        return False
    for p in range(min(n - 1, T - L) + 1):
        if all(a[(p + t) % n] == b[t % m] for t in range(L)):    # vẫn so L ký tự
            return True
    return False


print(match_offsets_full("abcd", "cdab", 2, 1))
print(match_offsets_full("abc", "bc", 2, 3))

True
False


# 5. Quan sát 2: so sánh `lcm(n, m)` ký tự là đủ

Chuỗi `a` lặp có chu kỳ `n`; chuỗi `b` lặp có chu kỳ `m`. Cả hai đều lặp lại sau mỗi `lcm(n, m)` ký tự (vì `lcm` là bội của cả `n` lẫn `m`):

```text
a[(p + t + lcm) % n] == a[(p + t) % n]
b[(t + lcm) % m]     == b[t % m]
```

Vậy nếu hai chuỗi khớp nhau trên `lcm(n, m)` ký tự đầu tiên, thì ký tự thứ `t` bất kỳ cũng khớp (đưa `t` về `t % lcm`). Ta chỉ cần so sánh:

```text
l = min(L, lcm(n, m))   ký tự
```

Với `n, m <= 100`, `lcm <= 9900`. Tổng công việc `<= 100 · 9900 ≈ 10^6` — chạy tốt!

In [6]:
def match_lcm(a: str, b: str, x: int, y: int) -> bool:
    n, m = len(a), len(b)
    L, T = m * y, n * x
    if L > T:
        return False
    l = min(L, lcm(n, m))
    for p in range(min(n - 1, T - L) + 1):
        if all(a[(p + t) % n] == b[t % m] for t in range(l)):
            return True
    return False


print(match_lcm("ab", "abab", 2, 1))                    # True
print(match_lcm("abc", "bc", 2, 3))                     # False
print(match_lcm("abcd", "cdab", 2, 1))                  # True
print(match_lcm("ab", "ba", 10**9, 10**9 - 1))          # x, y khổng lồ vẫn chạy tức thì

True
False
True
True


## Mô phỏng ví dụ 3: `a = "abcd"`, `b = "cdab"`, `x = 2`, `y = 1`

`n = m = 4`, `L = 4`, `T = 8`, `lcm = 4` → `l = 4`. Thử `p = 0..min(3, 4) = 0..3`:

| p | `a` lặp từ vị trí p (4 ký tự) | so với `"cdab"` |
|---:|---|:---:|
| 0 | `abcd` | ✗ |
| 1 | `bcda` | ✗ |
| 2 | `cdab` | ✔ |

Tìm thấy ở `p = 2` → `True`.

In [7]:
def trace_offsets(a: str, b: str, x: int, y: int, use_fw: bool = False) -> bool:
    n, m = len(a), len(b)
    L, T = m * y, n * x
    if L > T:
        print(f"L = {L} > T = {T} -> False")
        return False
    l = min(L, n + m - gcd(n, m)) if use_fw else min(L, lcm(n, m))
    pattern = "".join(b[t % m] for t in range(l))
    print(f"n = {n}, m = {m}, L = {L}, T = {T}, so sánh l = {l} ký tự, pattern = {pattern!r}")
    for p in range(min(n - 1, T - L) + 1):
        piece = "".join(a[(p + t) % n] for t in range(l))
        ok = piece == pattern
        print(f"  p = {p}: {piece!r} {'== khớp!' if ok else '!='}")
        if ok:
            return True
    return False


trace_offsets("abcd", "cdab", 2, 1)

n = 4, m = 4, L = 4, T = 8, so sánh l = 4 ký tự, pattern = 'cdab'
  p = 0: 'abcd' !=
  p = 1: 'bcda' !=
  p = 2: 'cdab' == khớp!


True

# 6. Quan sát 3 (mạnh hơn): định lý Fine–Wilf

Có một kết quả kinh điển trong lý thuyết chuỗi:

> **Định lý Fine–Wilf.** Nếu một chuỗi có **hai chu kỳ** `n` và `m` và độ dài của nó **ít nhất `n + m - gcd(n, m)`**, thì nó còn có chu kỳ `gcd(n, m)`.

Áp dụng: giả sử đoạn `l0 = n + m - gcd(n, m)` ký tự đầu của "`a` lặp từ `p`" trùng với `l0` ký tự đầu của "`b` lặp". Đoạn chung đó có chu kỳ `n` (vì là một phần của `a` lặp) **và** chu kỳ `m` (vì là phần của `b` lặp), độ dài đủ lớn → có chu kỳ `g = gcd(n, m)`. Từ đó suy ra `a` và `b` (khi xoay đúng pha) đều có chu kỳ `g` và trùng nhau mãi mãi.

Vậy chỉ cần so sánh:

```text
l = min(L, n + m - gcd(n, m))   ký tự      (<= 199)
```

Ta không chứng minh chi tiết định lý ở đây, nhưng sẽ **kiểm chứng bằng thực nghiệm** trên hàng chục nghìn bộ test ngẫu nhiên ở mục 11.

Ví dụ cho thấy ngưỡng `n + m - gcd` là **chặt**: với `n = 3`, `m = 2` thì ngưỡng là `4`. Chuỗi `"aba"` (độ dài 3 < 4) có chu kỳ 2 và chu kỳ 3 nhưng không có chu kỳ 1.

In [8]:
def has_period(s: str, q: int) -> bool:
    return all(s[i] == s[i + q] for i in range(len(s) - q))


s = "aba"
print(f"{s!r}: chu kỳ 2? {has_period(s, 2)}, chu kỳ 3? {has_period(s, 3)}, chu kỳ 1? {has_period(s, 1)}")

'aba': chu kỳ 2? True, chu kỳ 3? True, chu kỳ 1? False


In [9]:
def match_fine_wilf(a: str, b: str, x: int, y: int) -> bool:
    n, m = len(a), len(b)
    L, T = m * y, n * x
    if L > T:
        return False
    l = min(L, n + m - gcd(n, m))
    pattern = [b[t % m] for t in range(l)]
    for p in range(min(n - 1, T - L) + 1):
        if all(a[(p + t) % n] == pattern[t] for t in range(l)):
            return True
    return False


print(match_fine_wilf("ab", "abab", 2, 1))
print(match_fine_wilf("abc", "bc", 2, 3))
print(match_fine_wilf("abcd", "cdab", 2, 1))
trace_offsets("abc", "bc", 2, 3, use_fw=True)

True
False
True
n = 3, m = 2, L = 6, T = 6, so sánh l = 4 ký tự, pattern = 'bcbc'
  p = 0: 'abca' !=


False

# 7. Cách 4 — KMP: kiểm tra trong thời gian tuyến tính

Cách 3 thử `n` vị trí, mỗi vị trí so `l` ký tự → `O(n · (n + m))`. Có thể làm `O(n + m)`: dựng

```text
text    = (a lặp vô hạn)[0 : l + số_vị_trí - 1]
pattern = (b lặp vô hạn)[0 : l]
```

rồi tìm `pattern` trong `text` bằng **KMP**. Mọi vị trí bắt đầu `p` hợp lệ đều nằm trong `text`.

## 7.1. Mảng thất bại (failure function) của KMP

`f[i]` = độ dài **tiền tố riêng dài nhất** của `pattern[0..i]` cũng là **hậu tố** của nó.

Ví dụ `pattern = "abab"`: `f = [0, 0, 1, 2]` (ở `i = 3`, `"ab"` vừa là tiền tố vừa là hậu tố của `"abab"`).

Ý nghĩa: khi đang khớp được `k` ký tự mà gặp ký tự sai, ta **không quay lại từ đầu** mà nhảy về `f[k - 1]` — phần đã khớp vẫn còn tận dụng được.

In [10]:
def failure(pattern: str):
    f = [0] * len(pattern)
    k = 0
    for i in range(1, len(pattern)):
        while k and pattern[i] != pattern[k]:
            k = f[k - 1]                     # lùi về tiền tố-hậu tố ngắn hơn
        if pattern[i] == pattern[k]:
            k += 1
        f[i] = k
    return f


def kmp_find(text: str, pattern: str) -> int:
    f = failure(pattern)
    k = 0
    for i, ch in enumerate(text):
        while k and ch != pattern[k]:
            k = f[k - 1]
        if ch == pattern[k]:
            k += 1
        if k == len(pattern):
            return i - k + 1                 # vị trí bắt đầu của lần khớp đầu tiên
    return -1


print(failure("abab"), failure("aabaaab"))
print(kmp_find("abcdabcd", "cdab"), kmp_find("abcabc", "bcbcbc"))

[0, 0, 1, 2] [0, 1, 0, 1, 2, 2, 3]
2 -1


In [11]:
def match_kmp(a: str, b: str, x: int, y: int) -> bool:
    n, m = len(a), len(b)
    L, T = m * y, n * x
    if L > T:
        return False
    l = min(L, n + m - gcd(n, m))
    starts = min(n - 1, T - L) + 1
    text = "".join(islice(cycle(a), l + starts - 1))
    pattern = "".join(islice(cycle(b), l))
    return kmp_find(text, pattern) != -1


print(match_kmp("ab", "abab", 2, 1))
print(match_kmp("abc", "bc", 2, 3))
print(match_kmp("abcd", "cdab", 2, 1))

True
False
True


# 8. Phiên bản đúng format LeetCode

Ta nộp cách Fine–Wilf (cách 3): ngắn, dễ kiểm tra, và đủ nhanh (`<= 100 · 199` phép so sánh).

In [12]:
from math import gcd


class Solution:
    def repeatedStringMatch(self, a: str, b: str, x: int, y: int) -> bool:
        n, m = len(a), len(b)
        L, T = m * y, n * x
        if L > T:
            return False
        l = min(L, n + m - gcd(n, m))
        pattern = [b[t % m] for t in range(l)]
        for p in range(min(n - 1, T - L) + 1):
            if all(a[(p + t) % n] == pattern[t] for t in range(l)):
                return True
        return False

In [13]:
sol = Solution()
tests = [
    ("ab", "abab", 2, 1, True, "ví dụ 1"),
    ("abc", "bc", 2, 3, False, "ví dụ 2"),
    ("abcd", "cdab", 2, 1, True, "ví dụ 3"),
    ("a", "a", 1, 2, False, "b*y dài hơn a*x"),
    ("a", "a", 10**9, 10**9, True, "bằng nhau, cực lớn"),
    ("ab", "ba", 10**9, 10**9, False, "vừa khít độ dài nhưng lệch pha"),
    ("ab", "ba", 10**9, 10**9 - 1, True, "dư 2 ký tự để lệch pha"),
    ("abc", "cab", 1, 1, False, "xoay vòng nhưng a chỉ lặp 1 lần"),
]
for a, b, x, y, expected, name in tests:
    result = sol.repeatedStringMatch(a, b, x, y)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:34} -> {result}")

PASS | ví dụ 1                            -> True
PASS | ví dụ 2                            -> False
PASS | ví dụ 3                            -> True
PASS | b*y dài hơn a*x                    -> False
PASS | bằng nhau, cực lớn                 -> True
PASS | vừa khít độ dài nhưng lệch pha     -> False
PASS | dư 2 ký tự để lệch pha             -> True
PASS | xoay vòng nhưng a chỉ lặp 1 lần    -> False


Test `("ab", "ba", 10^9, 10^9)`: `a * x` và `b * y` dài bằng nhau, chỉ có vị trí `p = 0`, mà `"abab..."` khác `"baba..."` → `False`. Thêm một ký tự dư (`y = 10^9 - 1`) là có chỗ để lệch pha `p = 1` → `True`.

---

# 9. Vì sao đúng?

1. **Thu gọn vị trí:** khớp tại `p` chỉ phụ thuộc `p % n` (điều kiện ký tự) và `p + L <= T` (điều kiện độ dài, càng dễ khi `p` nhỏ). Nên chỉ cần thử `p <= min(n - 1, T - L)`.
2. **Thu gọn độ dài so sánh:** với chu kỳ chung `lcm`, khớp `lcm` ký tự đầu ⇒ khớp mọi ký tự. Định lý Fine–Wilf cho ngưỡng nhỏ hơn `n + m - gcd`. Nếu `L` nhỏ hơn ngưỡng thì ta so sánh đủ `L` ký tự — chính xác tuyệt đối.
3. **Không bỏ sót:** mọi vị trí hợp lệ đều được thử, mỗi lần thử là một phép so sánh đúng đắn.

---

## Trực giác

> Hai "dây đèn nhấp nháy" có chu kỳ `n` và `m`. Nếu chúng nhấp nháy giống nhau đủ lâu (đủ để mọi "lệch pha" có thể xảy ra đều đã xuất hiện), thì chúng sẽ giống nhau mãi mãi.

# 10. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | dựng chuỗi | `O(n·x · m·y)` | `O(n·x + m·y)` |
| 2. Thu gọn vị trí | `n` vị trí, so `L` ký tự | `O(n · m · y)` | `O(1)` |
| 2'. Thêm `lcm` | so `min(L, lcm)` ký tự | `O(n · lcm(n, m))` | `O(1)` |
| 3. Fine–Wilf | so `min(L, n + m - gcd)` ký tự | `O(n · (n + m))` | `O(n + m)` |
| 4. KMP | tìm mẫu trong văn bản ngắn | `O(n + m)` | `O(n + m)` |

# 11. Đối chiếu trên dữ liệu ngẫu nhiên

Dùng bảng chữ cái nhỏ (`"ab"`) và `x, y` nhỏ để brute force chạy được. Có 20.000 bộ test, trong đó nhiều bộ được sinh có chủ đích để `b` là một đoạn của `a` lặp.

In [14]:
import random

random.seed(4105)
count_true = 0
for _ in range(20000):
    a = "".join(random.choice("ab") for _ in range(random.randint(1, 5)))
    b = "".join(random.choice("ab") for _ in range(random.randint(1, 5)))
    if random.random() < 0.4:
        start = random.randint(0, len(a))
        b = (a * 4)[start:start + random.randint(1, 6)] or a
    x, y = random.randint(1, 8), random.randint(1, 8)
    expected = match_bruteforce(a, b, x, y)
    assert match_lcm(a, b, x, y) == expected, (a, b, x, y)
    assert match_fine_wilf(a, b, x, y) == expected, (a, b, x, y)
    assert match_kmp(a, b, x, y) == expected, (a, b, x, y)
    assert sol.repeatedStringMatch(a, b, x, y) == expected, (a, b, x, y)
    count_true += expected
print(f"20000 bộ test: mọi cách khớp brute force ({count_true} bộ True)")

20000 bộ test: mọi cách khớp brute force (3325 bộ True)


# 12. Những lỗi dễ mắc

## Lỗi 1 — Quên kiểm tra độ dài `L <= T`

Nếu bỏ điều kiện này, code có thể trả về `True` khi `b * y` **dài hơn** `a * x`.

In [15]:
def wrong_no_length_check(a, b, x, y):
    n, m = len(a), len(b)
    l = min(m * y, n + m - gcd(n, m))
    pattern = [b[t % m] for t in range(l)]
    return any(all(a[(p + t) % n] == pattern[t] for t in range(l)) for p in range(n))   # SAI


print("Sai :", wrong_no_length_check("a", "a", 1, 2))
print("Đúng:", sol.repeatedStringMatch("a", "a", 1, 2))

Sai : True
Đúng: False


## Lỗi 2 — Thử đủ `n` vị trí mà không chặn bởi `T - L`

Vị trí `p` phải thỏa `p + L <= T`. Thử `p` lớn hơn `T - L` có thể "khớp" nhưng thực chất tràn ra ngoài `a * x`.

In [16]:
def wrong_no_offset_limit(a, b, x, y):
    n, m = len(a), len(b)
    L, T = m * y, n * x
    if L > T:
        return False
    l = min(L, n + m - gcd(n, m))
    pattern = [b[t % m] for t in range(l)]
    return any(all(a[(p + t) % n] == pattern[t] for t in range(l)) for p in range(n))   # SAI


print("Sai :", wrong_no_offset_limit("ab", "ba", 1, 1))   # "ab" không chứa "ba"
print("Đúng:", sol.repeatedStringMatch("ab", "ba", 1, 1))

Sai : True
Đúng: False


## Lỗi 3 — So sánh quá ít ký tự (chỉ `max(n, m)` hoặc `m`)

So `m` ký tự chỉ kiểm tra **một** lần `b`, chưa đủ để đảm bảo các lần lặp sau vẫn khớp.

In [17]:
def wrong_compare_m(a, b, x, y):
    n, m = len(a), len(b)
    L, T = m * y, n * x
    if L > T:
        return False
    l = min(L, m)                                       # SAI: quá ngắn
    return any(all(a[(p + t) % n] == b[t % m] for t in range(l)) for p in range(min(n - 1, T - L) + 1))


random.seed(1)
for _ in range(10000):
    a = "".join(random.choice("ab") for _ in range(random.randint(1, 4)))
    b = "".join(random.choice("ab") for _ in range(random.randint(1, 4)))
    x, y = random.randint(1, 6), random.randint(1, 6)
    if wrong_compare_m(a, b, x, y) != match_bruteforce(a, b, x, y):
        print("Phản ví dụ:", (a, b, x, y), "| sai:", wrong_compare_m(a, b, x, y),
              "| đúng:", match_bruteforce(a, b, x, y))
        break

Phản ví dụ: ('ab', 'b', 4, 4) | sai: True | đúng: False


## Lỗi 4 — Dựng chuỗi thật khi `x, y` lớn

`a * x` với `x = 10^9` gây `MemoryError` hoặc chạy mãi. Luôn kiểm tra ràng buộc trước khi "nhân chuỗi".

## Lỗi 5 — Tràn số trong ngôn ngữ khác

`m · y` và `n · x` có thể tới `10^11` — vượt kiểu `int` 32-bit trong C++/Java. Python thì không sao.

# 13. Mẹo nhận diện

1. Chuỗi/mảng **lặp lại rất nhiều lần** (`10^9`) → không dựng ra; làm việc với **chỉ số modulo**.
2. Vị trí bắt đầu trong chuỗi tuần hoàn chu kỳ `n` → chỉ cần xét **`n` vị trí**.
3. So sánh hai chuỗi tuần hoàn → chỉ cần so một đoạn dài **`lcm`**, hoặc **`n + m - gcd`** (Fine–Wilf).
4. Tìm mẫu trong văn bản → **KMP** / **hàm Z** cho thời gian tuyến tính.

Các bài liên quan: *Repeated String Match* (LeetCode 686 — tìm số lần lặp `a` ít nhất để chứa `b`), *Repeated Substring Pattern* (459), *Find the Index of the First Occurrence in a String* (28).

# 14. Biến thể

## Biến thể 1 — LeetCode 686: lặp `a` ít nhất bao nhiêu lần để chứa `b`?

Dùng quan sát vị trí: nếu `b` xuất hiện, nó bắt đầu tại một `p < n`. Khi đó cần `a` lặp đủ để phủ `p + m` ký tự: `ceil((p + m) / n)` lần. Thử mọi `p` và lấy nhỏ nhất.

In [18]:
def repeated_string_match_686(a: str, b: str) -> int:
    n, m = len(a), len(b)
    best = -1
    for p in range(n):
        if all(a[(p + t) % n] == b[t] for t in range(m)):
            need = (p + m + n - 1) // n
            if best == -1 or need < best:
                best = need
    return best


print(repeated_string_match_686("abcd", "cdabcdab"))   # 3
print(repeated_string_match_686("a", "aa"))            # 2
print(repeated_string_match_686("abc", "wxyz"))        # -1

3
2
-1


## Biến thể 2 — `a` và `b` chỉ khớp sau khi "xoay"

Câu hỏi "`b` có phải là một phép xoay của `a` không?" (`|a| = |b|`) chính là trường hợp `x = 2, y = 1`: `b` là phép xoay của `a` ⇔ `b` nằm trong `a + a`.

In [19]:
def is_rotation(a: str, b: str) -> bool:
    return len(a) == len(b) and sol.repeatedStringMatch(a, b, 2, 1)


print(is_rotation("abcde", "cdeab"), is_rotation("abcde", "abced"))

True False


# 15. Template tư duy

```python
n, m = len(a), len(b)
L, T = m * y, n * x
if L > T: return False                       # 1. kiểm tra độ dài
l = min(L, n + m - gcd(n, m))                # 2. độ dài cần so (Fine–Wilf)
for p in range(min(n - 1, T - L) + 1):       # 3. chỉ n vị trí bắt đầu
    if all(a[(p + t) % n] == b[t % m] for t in range(l)):
        return True
return False
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

`a = "aba"`, `b = "ab"`, `x = 2`, `y = 2`. Tính `L`, `T`, ngưỡng `l`, các vị trí cần thử, và kết luận. Kiểm tra bằng cell dưới.

In [20]:
# Kiểm tra Bài 1
trace_offsets("aba", "ab", 2, 2, use_fw=True)

n = 3, m = 2, L = 4, T = 6, so sánh l = 4 ký tự, pattern = 'abab'
  p = 0: 'abaa' !=
  p = 1: 'baab' !=
  p = 2: 'aaba' !=


False

## Bài 2 — Dự đoán output

```python
print(gcd(12, 18), lcm(12, 18), 12 + 18 - gcd(12, 18))
print("".join(islice(cycle("xyz"), 8)))
print(failure("aaaa"), failure("abcab"))
```

In [21]:
print(gcd(12, 18), lcm(12, 18), 12 + 18 - gcd(12, 18))
print("".join(islice(cycle("xyz"), 8)))
print(failure("aaaa"), failure("abcab"))

6 36 24
xyzxyzxy
[0, 1, 2, 3] [0, 0, 0, 1, 2]


## Bài 3 — Hoàn thiện cách thu gọn vị trí + `lcm`

In [22]:
def exercise_3_match_lcm(a: str, b: str, x: int, y: int) -> bool:
    n, m = len(a), len(b)
    # TODO 1: tính L, T; nếu L > T thì False
    # TODO 2: l = min(L, lcm(n, m))
    # TODO 3: thử p từ 0 tới min(n - 1, T - L), so sánh l ký tự bằng chỉ số modulo
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_match_lcm("abcd", "cdab", 2, 1) == True
# assert exercise_3_match_lcm("abc", "bc", 2, 3) == False

## Bài 4 — Tự viết mảng thất bại KMP

```python
failure_exercise("abab")      -> [0, 0, 1, 2]
failure_exercise("aabaaab")   -> [0, 1, 0, 1, 2, 2, 3]
```

In [23]:
def exercise_4_failure(pattern: str):
    # TODO
    raise NotImplementedError


# assert exercise_4_failure("abab") == [0, 0, 1, 2]
# assert exercise_4_failure("aabaaab") == [0, 1, 0, 1, 2, 2, 3]

## Bài 5 — Repeated Substring Pattern (LeetCode 459)

Chuỗi `s` có tạo được bằng cách lặp một chuỗi con thực sự (ít nhất 2 lần) không?

```python
repeated_pattern("abab")   -> True     # "ab" * 2
repeated_pattern("aba")    -> False
repeated_pattern("abcabcabcabc") -> True
```

Gợi ý: dùng mảng thất bại: `p = n - f[n-1]` là chu kỳ nhỏ nhất.

In [24]:
def exercise_5_repeated_pattern(s: str) -> bool:
    # TODO
    raise NotImplementedError


# assert exercise_5_repeated_pattern("abab") == True
# assert exercise_5_repeated_pattern("aba") == False
# assert exercise_5_repeated_pattern("abcabcabcabc") == True

## Bài 6 — Mini challenge: đếm số lần xuất hiện

Đếm **số vị trí** `p` mà `b * y` xuất hiện trong `a * x` (có thể tới `~10^11`, không được liệt kê).

```python
count_occurrences("ab", "ab", 3, 1)          -> 3       # "ababab": vị trí 0, 2, 4
count_occurrences("a", "a", 10**9, 1)        -> 10**9
count_occurrences("abc", "bc", 2, 3)         -> 0
```

Gợi ý: khớp tại `p` chỉ phụ thuộc `r = p % n`. Với mỗi `r` khớp, đếm số `p ≡ r (mod n)` trong `[0, T - L]`.

In [25]:
def exercise_6_count_occurrences(a: str, b: str, x: int, y: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_6_count_occurrences("ab", "ab", 3, 1) == 3
# assert exercise_6_count_occurrences("a", "a", 10**9, 1) == 10**9
# assert exercise_6_count_occurrences("abc", "bc", 2, 3) == 0

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `L = m * y`, `T = n * x`.
- Mức 2: ký tự thứ `t` của "a lặp từ p" là `a[(p + t) % n]`, của "b lặp" là `b[t % m]`.
- Mức 3: `return True` ngay khi một `p` khớp đủ `l` ký tự; hết vòng lặp thì `False`.

**Bài 4**

- Mức 1: `k` là độ dài khớp hiện tại, bắt đầu `0`.
- Mức 2: khi `pattern[i] != pattern[k]`, lùi `k = f[k - 1]` (lặp lại).
- Mức 3: nếu bằng thì `k += 1`; gán `f[i] = k`.

**Bài 5**

- Mức 1: tính `f` của `s`.
- Mức 2: `p = n - f[n - 1]`.
- Mức 3: đúng khi `f[n - 1] > 0` và `n % p == 0`.

**Bài 6**

- Mức 1: nếu `L > T` thì `0`.
- Mức 2: với mỗi `r` trong `0..n-1` khớp (so `l` ký tự), số `p = r, r + n, r + 2n, ... <= T - L` là `(T - L - r) // n + 1` nếu `r <= T - L`.
- Mức 3: cộng lại.

# 17. Đáp án tham khảo

## Đáp án Bài 1

`n = 3`, `m = 2`, `L = 4`, `T = 6`, `gcd = 1` → `l = min(4, 3 + 2 - 1) = 4`, pattern `"abab"`. Thử `p = 0..min(2, 2)`:

| p | 4 ký tự của `a` lặp từ p | khớp `"abab"`? |
|---:|---|:---:|
| 0 | `abaa` | ✗ |
| 1 | `baab` | ✗ |
| 2 | `aaba` | ✗ |

→ `False`. (Thật vậy, `"abaaba"` không chứa `"abab"`.)

## Đáp án Bài 2

```text
6 36 24
xyzxyzxy
[0, 1, 2, 3] [0, 0, 0, 1, 2]
```

In [26]:
# Đáp án Bài 3
def answer_3_match_lcm(a: str, b: str, x: int, y: int) -> bool:
    n, m = len(a), len(b)
    L, T = m * y, n * x
    if L > T:
        return False
    l = min(L, lcm(n, m))
    for p in range(min(n - 1, T - L) + 1):
        if all(a[(p + t) % n] == b[t % m] for t in range(l)):
            return True
    return False


assert answer_3_match_lcm("abcd", "cdab", 2, 1) == True
assert answer_3_match_lcm("abc", "bc", 2, 3) == False
print("Bài 3: PASS")

Bài 3: PASS


In [27]:
# Đáp án Bài 4
def answer_4_failure(pattern: str):
    f = [0] * len(pattern)
    k = 0
    for i in range(1, len(pattern)):
        while k and pattern[i] != pattern[k]:
            k = f[k - 1]
        if pattern[i] == pattern[k]:
            k += 1
        f[i] = k
    return f


assert answer_4_failure("abab") == [0, 0, 1, 2]
assert answer_4_failure("aabaaab") == [0, 1, 0, 1, 2, 2, 3]
print("Bài 4: PASS")

Bài 4: PASS


In [28]:
# Đáp án Bài 5
def answer_5_repeated_pattern(s: str) -> bool:
    f = answer_4_failure(s)
    n = len(s)
    p = n - f[-1]
    return f[-1] > 0 and n % p == 0


assert answer_5_repeated_pattern("abab") == True
assert answer_5_repeated_pattern("aba") == False
assert answer_5_repeated_pattern("abcabcabcabc") == True
for s in ["a", "aa", "ab", "abcab", "aabaab", "abac"]:
    assert answer_5_repeated_pattern(s) == (s in (s + s)[1:-1])
print("Bài 5: PASS")

Bài 5: PASS


In [29]:
# Đáp án Bài 6
def answer_6_count_occurrences(a: str, b: str, x: int, y: int) -> int:
    n, m = len(a), len(b)
    L, T = m * y, n * x
    if L > T:
        return 0
    l = min(L, n + m - gcd(n, m))
    total = 0
    for r in range(min(n - 1, T - L) + 1):
        if all(a[(r + t) % n] == b[t % m] for t in range(l)):
            total += (T - L - r) // n + 1
    return total


def brute_count(a, b, x, y):
    big, pat = a * x, b * y
    return sum(1 for p in range(len(big) - len(pat) + 1) if big[p:p + len(pat)] == pat)


assert answer_6_count_occurrences("ab", "ab", 3, 1) == 3
assert answer_6_count_occurrences("a", "a", 10**9, 1) == 10**9
assert answer_6_count_occurrences("abc", "bc", 2, 3) == 0
random.seed(6)
for _ in range(3000):
    a = "".join(random.choice("ab") for _ in range(random.randint(1, 4)))
    b = "".join(random.choice("ab") for _ in range(random.randint(1, 4)))
    x, y = random.randint(1, 6), random.randint(1, 6)
    assert answer_6_count_occurrences(a, b, x, y) == brute_count(a, b, x, y), (a, b, x, y)
print("Bài 6: PASS (kể cả 3000 test ngẫu nhiên)")

Bài 6: PASS (kể cả 3000 test ngẫu nhiên)


# 18. Tổng kết

**Repeated String Match II** là bài học về **tính tuần hoàn**:

1. Kiểm tra độ dài `m·y <= n·x`.
2. Vị trí bắt đầu chỉ cần xét `p < n` (và `p <= n·x - m·y`).
3. Chỉ cần so `min(m·y, n + m - gcd(n, m))` ký tự (Fine–Wilf), hoặc an toàn hơn `min(m·y, lcm(n, m))`.
4. Có thể thay vòng thử bằng KMP để đạt `O(n + m)`.

## Độ phức tạp của lời giải nộp

```text
Time:  O(n · (n + m))   (<= 100 · 199)
Space: O(n + m)
```

# 19. Checklist trước khi submit

- [ ] Đã kiểm tra `m * y <= n * x` trước tiên?
- [ ] Vị trí bắt đầu chặn bởi `min(n - 1, n*x - m*y)`?
- [ ] Độ dài so sánh `min(m*y, n + m - gcd)` (hoặc `lcm`), không phải chỉ `m`?
- [ ] Không bao giờ dựng `a * x` hay `b * y` khi `x, y` lớn?
- [ ] Đã test: độ dài vừa khít, lệch pha, `x` hoặc `y` cực lớn, `a` một ký tự?